РЕГРЕССИЯ — BASELINE

Цель: Обучить 5 моделей для 3 целевых переменных (IC50, CC50, SI)
и оценить их baseline-производительность.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Добавляем корень проекта в путь
sys.path.append(os.path.dirname(os.getcwd()))

from src import get_regression_models
from src import evaluate_regression

In [17]:
SAVE_PATH = '../data/processed/'

X_train = joblib.load(f'{SAVE_PATH}X_train_scaled.pkl')
X_test = joblib.load(f'{SAVE_PATH}X_test_scaled.pkl')
y_train_reg = joblib.load(f'{SAVE_PATH}y_train_reg.pkl')
y_test_reg = joblib.load(f'{SAVE_PATH}y_test_reg.pkl')

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")


X_train: (798, 165), X_test: (200, 165)


## Определение целевых переменных

Из загруженных данных мы извлекаем список регрессионных таргетов, которые будем прогнозировать:

- **pIC50** - отрицательный десятичный логарифм IC50 (ингибирующая концентрация). Чем выше значение, тем активнее соединение.
- **pCC50** - отрицательный десятичный логарифм CC50 (цитотоксическая концентрация). Чем выше значение, тем менее токсично соединение.
- **log_SI** - логарифм индекса селективности (SI = CC50/IC50). Чем выше значение, тем более селективно соединение (лучше различает здоровые и инфицированные клетки).

Для каждой из этих переменных мы обучим 5 моделей и сравним их производительность.

In [7]:
REGRESSION_TARGETS = y_train_reg.columns.tolist()
REGRESSION_TARGETS

['pIC50', 'pCC50', 'log_SI']

## Инициализация моделей

Для baseline-экспериментов мы используем 5 моделей регрессии, каждая из которых имеет свои преимущества:

1. **LinearRegression** - базовая линейная модель для проверки линейности данных
2. **DecisionTree** - простое дерево решений для выявления нелинейных зависимостей
3. **RandomForest** - ансамбль деревьев, устойчивый к переобучению
4. **GradientBoosting** - градиентный бустинг, последовательно улучшающий предсказания
5. **XGBoost** - оптимизированная реализация градиентного бустинга с регуляризацией

Все модели инициализируются с параметрами по умолчанию для получения чистого baseline.

In [8]:
models = get_regression_models()

# Структура для результатов
results = {
    'IC50': {},
    'CC50': {},
    'SI': {}
}

## Обучение и оценка моделей

В этом блоке происходит основной цикл обучения и оценки моделей для каждого таргета:

**Процесс:**
1. Для каждого таргета (pIC50, pCC50, log_SI) извлекаются соответствующие y_train и y_test
2. Каждая модель обучается на тренировочных данных с помощью метода `.fit()`
3. Выполняется предсказание на тестовых данных через `.predict()`
4. Вычисляются метрики качества с помощью функции `evaluate_regression`

**Метрики оценки:**
- **RMSE** (Root Mean Square Error) - среднеквадратичная ошибка, чувствительна к выбросам
- **MAE** (Mean Absolute Error) - средняя абсолютная ошибка, интерпретируема в единицах таргета
- **R²** (коэффициент детерминации) - доля объяснённой дисперсии, от 0 до 1 (чем выше, тем лучше)

In [9]:
for target in REGRESSION_TARGETS:
    print(f"ЦЕЛЕВАЯ ПЕРЕМЕННАЯ: {target}")
    print('─'*60)
    
    y_train = y_train_reg[target]
    y_test = y_test_reg[target]
    
    for name, model in models.items():
        # Обучение
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Оценка
        metrics = evaluate_regression(y_test, y_pred)
        
        # Сохранение
        target_key = 'IC50' if 'pIC50' in target else 'CC50' if 'pCC50' in target else 'SI'
        results[target_key][name] = metrics
        
        # Вывод
        print(f"{name:20} | RMSE: {metrics['RMSE']:.4f} | MAE: {metrics['MAE']:.4f} | R²: {metrics['R2']:.4f}")

ЦЕЛЕВАЯ ПЕРЕМЕННАЯ: pIC50
────────────────────────────────────────────────────────────
LinearRegression     | RMSE: 1195.0314 | MAE: 85.1499 | R²: -1388224.5222
DecisionTree         | RMSE: 0.8823 | MAE: 0.6308 | R²: 0.2433
RandomForest         | RMSE: 0.7096 | MAE: 0.5430 | R²: 0.5106
GradientBoosting     | RMSE: 0.7019 | MAE: 0.5565 | R²: 0.5211
XGBoost              | RMSE: 0.7376 | MAE: 0.5430 | R²: 0.4711
ЦЕЛЕВАЯ ПЕРЕМЕННАЯ: pCC50
────────────────────────────────────────────────────────────
LinearRegression     | RMSE: 584.0117 | MAE: 41.7547 | R²: -690025.2853
DecisionTree         | RMSE: 0.6994 | MAE: 0.4421 | R²: 0.0105
RandomForest         | RMSE: 0.5529 | MAE: 0.3751 | R²: 0.3816
GradientBoosting     | RMSE: 0.5647 | MAE: 0.3917 | R²: 0.3549
XGBoost              | RMSE: 0.5827 | MAE: 0.3847 | R²: 0.3130
ЦЕЛЕВАЯ ПЕРЕМЕННАЯ: log_SI
────────────────────────────────────────────────────────────
LinearRegression     | RMSE: 611.0203 | MAE: 43.7862 | R²: -540848.5452
DecisionTree    

## Итоговые результаты по таргетам

**Анализ результатов:**

**Для pIC50:**
- Лучшие результаты показывают GradientBoosting (R² = 0.5211) и RandomForest (R² = 0.5106)
- Линейная регрессия даёт катастрофический результат (R² ≈ -1.4 млн), что указывает на сильную нелинейность

**Для pCC50:**
- RandomForest показывает лучший результат (R² = 0.3816)
- Все модели работают хуже, чем для pIC50, что говорит о более сложной природе цитотоксичности

**Для log_SI:**
- RandomForest - лучший (R² = 0.3057), но результат скромный
- Прогнозирование селективности оказалось самой сложной задачей

In [13]:
# Создаём таблицы для каждого таргета
for target_key in ['IC50', 'CC50', 'SI']:
    print(f"{target_key} — Результаты")
    print('─'*60)
    
    df = pd.DataFrame(results[target_key]).T
    display(df.round(4))
    
    # Сохраняем
    df.to_csv(f'../artifacts/results/regression_baseline_{target_key}.csv')

IC50 — Результаты
────────────────────────────────────────────────────────────


,RMSE,MAE,R2
LinearRegression,1195.0314,85.1499,-1.388225e+06
DecisionTree,0.8823,0.6308,2.433000e-01
RandomForest,0.7096,0.5430,5.106000e-01
GradientBoosting,0.7019,0.5565,5.211000e-01
XGBoost,0.7376,0.5430,4.711000e-01


CC50 — Результаты
────────────────────────────────────────────────────────────


,RMSE,MAE,R2
LinearRegression,584.0117,41.7547,-690025.2853
DecisionTree,0.6994,0.4421,0.0105
RandomForest,0.5529,0.3751,0.3816
GradientBoosting,0.5647,0.3917,0.3549
XGBoost,0.5827,0.3847,0.3130


SI — Результаты
────────────────────────────────────────────────────────────


,RMSE,MAE,R2
LinearRegression,611.0203,43.7862,-540848.5452
DecisionTree,0.8254,0.6015,0.0130
RandomForest,0.6923,0.5078,0.3057
GradientBoosting,0.7107,0.5232,0.2682
XGBoost,0.7017,0.5000,0.2867


**Основные наблюдения:**
- **GradientBoosting** и **RandomForest** стабильно входят в топ-2 для всех трёх таргетов
- **XGBoost** показывает конкурентоспособные результаты, особенно для IC50 и SI
- **DecisionTree** значительно уступает ансамблевым методам
- **LinearRegression** неприменима для этих данных из-за сильной нелинейности


In [11]:
summary_r2 = pd.DataFrame({
    'IC50': [results['IC50'][name]['R2'] for name in models.keys()],
    'CC50': [results['CC50'][name]['R2'] for name in models.keys()],
    'SI': [results['SI'][name]['R2'] for name in models.keys()]
}, index=models.keys())

display(summary_r2.round(4))

# Сохраняем сводную таблицу
summary_r2.to_csv('../artifacts/results/regression_baseline_summary.csv')

,IC50,CC50,SI
LinearRegression,-1.388225e+06,-690025.2853,-540848.5452
DecisionTree,2.433000e-01,0.0105,0.0130
RandomForest,5.106000e-01,0.3816,0.3057
GradientBoosting,5.211000e-01,0.3549,0.2682
XGBoost,4.711000e-01,0.3130,0.2867


## Выбор топ-моделей для оптимизации

На основе полученных результатов мы выбираем по две лучшие модели для каждого таргета

**Критерий выбора:** максимальное значение R² на тестовой выборке

Выбранные модели сохраняются в файл `top_models_regression.pkl` для использования в следующем ноутбуке.

In [14]:
top_models = {}

for target_key in ['IC50', 'CC50', 'SI']:
    # Сортируем по R²
    sorted_models = sorted(
        results[target_key].items(),
        key=lambda x: x[1]['R2'],
        reverse=True
    )
    top_2 = [name for name, _ in sorted_models[:2]]
    top_models[target_key] = top_2
    
    print(f"\n{target_key}:")
    for i, name in enumerate(top_2, 1):
        print(f"  {i}. {name} (R² = {results[target_key][name]['R2']:.4f})")

# Сохраняем топ-модели
joblib.dump(top_models, '../artifacts/top_models_regression.pkl')


IC50:
  1. GradientBoosting (R² = 0.5211)
  2. RandomForest (R² = 0.5106)

CC50:
  1. RandomForest (R² = 0.3816)
  2. GradientBoosting (R² = 0.3549)

SI:
  1. RandomForest (R² = 0.3057)
  2. XGBoost (R² = 0.2867)


['../artifacts/top_models_regression.pkl']